In [1]:
import json

import chromadb
from sentence_transformers import SentenceTransformer

articles = json.load(open("../data/processed/civil_code.json"))
by_num = {a["article_number"]: a for a in articles}
rows = [a for a in articles if a["text_ar"].strip()]
assert len({a["article_id"] for a in rows}) == len(rows)

model = SentenceTransformer("intfloat/multilingual-e5-small")
embs = model.encode(
    [f"passage: {a['text_ar']}" for a in rows],
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
)

col = chromadb.EphemeralClient().create_collection("civil_code", metadata={"hnsw:space": "cosine"})
col.add(
    ids=[a["article_id"] for a in rows],
    embeddings=embs.tolist(),
    documents=[a["text_ar"] for a in rows],
    metadatas=[{"article_number": a["article_number"]} for a in rows],
)
print(col.count())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

1149


In [2]:
def search(q: str, k: int = 3) -> list[int]:
    e = model.encode([f"query: {q}"], normalize_embeddings=True)
    r = col.query(query_embeddings=e.tolist(), n_results=k)
    return [m["article_number"] for m in r["metadatas"][0]]

In [3]:
eval_set = [
    ("ماذا يطبق القاضي إذا لم يوجد نص تشريعي؟", 1),
    ("What does the judge apply if there is no applicable legislative provision?", 1),
    ("هل يجوز لأحد المتعاقدين تعديل العقد بإرادته المنفردة؟", 147),
    ("Can one party unilaterally modify a contract?", 147),
    ("هل يجب تنفيذ العقد بحسن نية؟", 148),
    ("Must a contract be performed in good faith?", 148),
    ("ما التزام من يتسبب بخطئه في ضرر للغير؟", 163),
    ("What is the obligation of a person whose fault causes harm to another?", 163),
    ("ما هو سن الرشد؟", 44),
    ("What is the age of majority?", 44),
]

In [4]:
for n in sorted({g for _, g in eval_set}):
    print(n, by_num[n]["text_en"][:150], "\n")

1 Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision of a law that is applicable,  

44 All persons attaining majority in possession of their mental faculties and not under legal disability, have full legal capacity to exercise their civi 

147 The contract makes the law of the parties. It can be revoked or altered only by mutual consent of the parties or for reasons provided for by law. When 

148 A contract must be performed in accordance with its contents and in compliance with the requirements of good faith. A contract binds the contracting p 

163 Every fault which cause injury to another, imposes an obligation to make reparation upon the person by whom it is committed. 



In [5]:
hits = 0
for q, gold in eval_set:
    got = search(q)
    ok = gold in got
    hits += ok
    print("✅" if ok else "❌", gold, got, q)
print("hit@3 =", hits / len(eval_set))

✅ 1 [1, 200, 2] ماذا يطبق القاضي إذا لم يوجد نص تشريعي؟
✅ 1 [27, 1086, 1] What does the judge apply if there is no applicable legislative provision?
✅ 147 [147, 154, 453] هل يجوز لأحد المتعاقدين تعديل العقد بإرادته المنفردة؟
✅ 147 [672, 147, 857] Can one party unilaterally modify a contract?
✅ 148 [148, 124, 1085] هل يجب تنفيذ العقد بحسن نية؟
✅ 148 [124, 148, 512] Must a contract be performed in good faith?
❌ 163 [168, 166, 521] ما التزام من يتسبب بخطئه في ضرر للغير؟
✅ 163 [163, 168, 311] What is the obligation of a person whose fault causes harm to another?
✅ 44 [44, 46, 621] ما هو سن الرشد؟
✅ 44 [44, 45, 518] What is the age of majority?
hit@3 = 0.9


In [22]:
import os

from dotenv import find_dotenv, load_dotenv
from groq import Groq

load_dotenv(find_dotenv(), override=True)
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "qwen/qwen3.8-27b"  # انسخي اسم موديل حالي من console.groq.com/docs/models

SYSTEM = (
    "أنت مساعد قانوني. أجب بالعربية الفصحى وبالاعتماد فقط على المواد المرفقة. "
    "اذكر رقم المادة التي اعتمدت عليها. إذا لم تحتوِ المواد على الإجابة فقل ذلك صراحة."
)


def answer(q: str, k: int = 3) -> str:
    nums = search(q, k)
    ctx = "\n\n".join(f"المادة {n}: {by_num[n]['text_ar']}" for n in nums)
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        max_completion_tokens=400,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"المواد:\n{ctx}\n\nالسؤال: {q}"},
        ],
    )
    return resp.choices[0].message.content


print(answer("ما هو سن الرشد؟"))

بناءً على المادة 44، فقرة (2)، فإن سن الرشد هي **إحدى وعشرون سنة ميلادية كاملة**.


In [23]:
print(by_num[44]["text_ar"])

(1) كل شخص بلغ سن الرشد متمتعا بقواه العقلية ، ولم يحجر عليه ، يكون كامل الأهلية لمباشرة حقوقه المدنية. (2) وسن الرشد هى إحدى وعشرون سنة ميلادية كاملة.


In [24]:
import time

for q in [
    "What is the age of majority?",
    "ما التزام من يتسبب بخطئه في ضرر للغير؟",
    "ما هي عاصمة فرنسا؟",
]:
    print(q, "\n", answer(q), "\n", "-" * 40)
    time.sleep(10)

What is the age of majority? 
 بناءً على المادة 44، فإن سن الرشد هي **إحدى وعشرون سنة ميلادية كاملة**. 
 ----------------------------------------
ما التزام من يتسبب بخطئه في ضرر للغير؟ 
 بناءً على المواد المرفقة، لا يوجد نص صريح يحدد التزام من يتسبب بخطئه في ضرر للغير في الحالة العامة (أي خارج حالات الدفاع الشرعي أو الضرورة).

المواد المرفقة تتناول حالات استثنائية أو خاصة:
*   **المادة 166**: تنص على عدم المسؤولية في حالة الدفاع الشرعي، مع وجوب التعويض إذا جاوز المدافع القدر الضروري.
*   **المادة 168**: تنص على أن من سبب ضررا ليتفادى ضررا أكبر محدقا، لا يكون ملزما إلا بالتعويض الذي يراه القاضي مناسبا.
*   **المادة 521**: تتناول التزامات الشريك تجاه الشركة، وليس المسؤولية المدنية العامة عن الخطأ.

لذلك، **لا تحتوي المواد المرفقة على إجابة صريحة** تسأل عن التزام من يتسبب بخطئه في ضرر للغير في الحالة العامة (أي خارج نطاق الدفاع الشرعي أو الضرورة). 
 ----------------------------------------
ما هي عاصمة فرنسا؟ 
 لا تحتوي المواد المرفقة على إجابة لسؤالك، حيث تتناول هذه المواد أحكامًا قانونية